# data-profiler — профиль + синтетический сэмпл (закрытый контур)

Одна ячейка ниже: **параметры + запуск**. Вся логика — в пакете `profiler/`.

Результат в `OUTPUT_DIR`: `profiles/<schema>.<table>.profile.json` (типы, min/max, все категории, кардинальность, null%, PK-гипотеза) и `samples/<schema>.<table>.sample.csv` (синтетические строки).

In [ ]:
# ── Подключение к БД (SQLAlchemy) ────────────────────────────────────────────
# Либо DB_URL целиком, либо по частям. Пароль/порт опциональны.
DB_URL = "postgresql+psycopg2://user@host:5432/db"
# DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME = "user", "", "host", 5432, "db"

# ── По каким таблицам собирать (schema.table) ────────────────────────────────
TABLES = [
    "sn_uzp.task",         # описания подтянутся из парной sn_view.task (redirect-заглушка)
    "sn_t_uzp.task_log",   # sn_t_uzp читает комментарии сам из себя
]

# Справочники: выгружать ЦЕЛИКОМ (все строки, реальные значения), маскируя ТОЛЬКО
# персональные поля (ФИО/логины). Коды и id (tb_id, gosb_id …) остаются как есть.
FULL_TABLES = [
    # "sn_uzp.dict_tb",
]

# ── Корреляционные группы: значения внутри группы НЕ перемешиваются между строк
# (напр. результат опросника остаётся «своим» для своего подтипа задачи)
CORRELATED_GROUPS = [
    ["task_type", "task_subtype", "task_questionary"],
]

# ── Оверрайды чувствительности (приоритет над авто-эвристикой) ────────────────
# НЕ маскировать (наименования справочников/объектов, а не персональные данные):
NON_SENSITIVE_COLUMNS = ["tb_full_name"]
# Принудительно маскировать: {"колонка": "inn|fio|phone|email|account|money|address|freeform|..."}
SENSITIVE_COLUMNS = {}

# ── Объёмы ───────────────────────────────────────────────────────────────────
MAX_CATEGORIES      = 300      # если уник. значений <= этого (и поле не чувствит.) — в профиль попадут ВСЕ значения категории
SAMPLE_ROWS_PROFILE = 100000   # сколько строк тянуть из БД в pandas для профиля (по нему считаются кардинальность/null%/PK)
SYNTH_ROWS          = 1000     # сколько синтетических строк сгенерировать в сэмпл-CSV
OUTPUT_DIR          = "./output"

# LLM_POOL_SIZE — сколько РАЗНЫХ фейковых значений LLM придумывает на одно free-text поле
#   (напр. на подтип задачи); строки потом набираются повтором из этого пула.
#   Больше = разнообразнее, но дороже по токенам. На категории/ИНН не влияет.
LLM_POOL_SIZE       = 60
# SEED — фиксирует генератор случайных чисел: повторный прогон даёт тот же сэмпл.
SEED                = 42

# ── LLM ───────────────────────────────────────────────────────────────────────
# method — ТРАНСПОРТ (не модель): "http" (requests) | "gigachat" (langchain) | None.
# model  — любая модель шлюза (Qwen3.5-397b / glm-5.1 / …), в любом транспорте.
# Токены и URL берутся из .env автоматически (файл в .gitignore, вводить не нужно).

# Закрытый контур (env GIGACHAT_API_URL / JPY_API_TOKEN — дефолт):
LLM = dict(method="http", model="Qwen3.5-397b")
# LLM = dict(method="gigachat", model="Qwen3.5-397b")   # альтернативный транспорт
# LLM = dict(method="http", model="glm-5.1")            # другая модель

# Тест на DeepSeek (ключ уже лежит в .env):
# LLM = dict(method="http", model="deepseek-chat",
#            base_url_env="DEEPSEEK_BASE_URL", token_env="DEEPSEEK_API_KEY")

# LLM = dict(method=None)  # полностью офлайн: синтез через детерминированный фейкер

# ── Запуск ───────────────────────────────────────────────────────────────────
from profiler import run
result = run(locals())
result